In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog qiskit-serverless
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# 첫 번째 Qiskit Serverless 프로그램 작성하기

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하길 권장합니다.

```
qiskit[all]~=1.3.1
qiskit-ibm-runtime~=0.34.0
qiskit-aer~=0.15.1
qiskit-serverless~=0.18.1
qiskit-ibm-catalog~=0.2
qiskit-addon-sqd~=0.8.1
qiskit-addon-utils~=0.1.0
qiskit-addon-mpf~=0.2.0
qiskit-addon-aqc-tensor~=0.1.2
qiskit-addon-obp~=0.1.0
scipy~=1.15.0
pyscf~=2.8.0
```
</details>
이 예제는 `qiskit-serverless` 도구를 사용하여 병렬 트랜스파일 프로그램을 생성하고, `qiskit-ibm-catalog`를 구현하여 프로그램을 IBM Quantum Platform에 배포함으로써 재사용 가능한 원격 서비스로 활용하는 방법을 설명합니다.
## 예제: Qiskit Serverless를 이용한 원격 Transpiler
먼저 다음 예제를 살펴보겠습니다. 이 예제는 주어진 `backend`와 대상 `optimization_level`에 대해 `circuit`을 트랜스파일하며, 워크로드를 Qiskit Serverless에 배포하기 위한 요소들을 점진적으로 추가합니다.

다음 코드 셀을 `./source_files/transpile_remote.py` 파일에 넣어 주세요. 이 파일이 Qiskit Serverless에 업로드할 프로그램입니다.

In [ ]:
# This cell is hidden from users, it creates a new folder
from pathlib import Path

Path("./source_files").mkdir(exist_ok=True)

In [ ]:
%%writefile ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit.transpiler import generate_preset_pass_manager

def transpile_remote(circuit, optimization_level, backend):
    """Transpiles an abstract circuit into an ISA circuit for a given backend."""
    pass_manager = generate_preset_pass_manager(
        optimization_level=optimization_level,
		backend=backend
    )
    isa_circuit = pass_manager.run(circuit)
    return isa_circuit

### Add code to get program arguments

Now add the following code to your program file, which sets up program arguments.

Your initial `transpile_remote.py` has three inputs: `circuits`, `backend_name`, and `optimization_level`. Serverless is currently limited to only accept serializable inputs and outputs. For this reason, you cannot pass in `backend` directly, so use `backend_name` as a string instead.

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit_serverless import get_arguments, save_result, distribute_task, get

# Get program arguments
arguments = get_arguments()
circuits = arguments.get("circuits")
backend_name = arguments.get("backend_name")
optimization_level = arguments.get("optimization_level")

## 파일 설정하기
Qiskit Serverless를 사용하려면 워크로드의 `.py` 파일을 전용 디렉토리에 설정해야 합니다. 다음 구조는 모범 사례의 예시입니다.

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend(backend_name)

Serverless는 `source_files`의 내용을 업로드하여 원격으로 실행합니다. 설정이 완료되면 `transpile_remote.py`를 수정하여 입력을 가져오고 출력을 반환할 수 있습니다.

### 프로그램 인수 가져오기
초기 `transpile_remote.py`에는 `circuits`, `backend_name`, `optimization_level` 세 가지 입력이 있습니다. Serverless는 현재 직렬화 가능한 입력 및 출력만 허용하도록 제한되어 있습니다. 이러한 이유로 `backend`를 직접 전달할 수 없으므로, 대신 `backend_name`을 문자열로 사용합니다.

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

# Each circuit is being transpiled and will populate the array
results = [
    transpile_remote(circuit, 1, backend)
    for circuit in circuits
]

save_result({
    "transpiled_circuits": results
})

<span id="upload-to-qiskit-serverless"></span>
### Authenticate to Qiskit Serverless

Use `qiskit-ibm-catalog` to authenticate to `QiskitServerless` with your API key (you can use your `QiskitRuntimeService` API key, or create a new API key on the [IBM Quantum Platform dashboard](https://quantum.cloud.ibm.com)).

In [13]:
from qiskit_ibm_catalog import QiskitServerless, QiskitFunction

# Authenticate to the remote cluster and submit the pattern for remote execution
serverless = QiskitServerless()

이 시점에서 `QiskitRuntimeService`를 사용하여 Backend를 가져오고, 다음 코드로 기존 프로그램을 추가할 수 있습니다. 아래 코드를 실행하려면 [자격 증명을 저장](/guides/cloud-setup)해 두어야 합니다.

In [8]:
transpile_remote_demo = QiskitFunction(
    title="transpile_remote_serverless",
    entrypoint="transpile_remote.py",
    working_dir="./source_files/",
)

In [9]:
serverless.upload(transpile_remote_demo)

QiskitFunction(transpile_remote_serverless)

마지막으로, 전달된 모든 `circuits`에 대해 `transpile_remote()`를 실행하고 `transpiled_circuits`를 결과로 반환할 수 있습니다.

In [10]:
# Get program from serverless.list() that matches the title of the one we uploaded
next(
    program
    for program in serverless.list()
    if program.title == "transpile_remote_serverless"
)

QiskitFunction(transpile_remote_serverless)

In [11]:
# This cell is hidden from users, it checks the program uploaded correctly
assert _.title == "transpile_remote_serverless"  # noqa: F821

In [12]:
# This cell is hidden from users, it checks the program executes correctly
from time import sleep
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
transpile_remote_serverless = serverless.load("transpile_remote_serverless")
job = transpile_remote_serverless.run(
    circuits=[qc],
    backend="ibm_sherbrooke",
    optimization_level=1,
)
while True:
    sleep(5)
    status = job.status()
    if status not in ["QUEUED", "INITIALIZING", "RUNNING", "DONE"]:
        raise Exception(
            f"Unexpected job status: '{status}'\n"
            + "Here are the logs:\n"
            + job.logs()
        )
    if status == "DONE":
        break

Qiskit Serverless는 `working_dir`의 내용(이 경우 `source_files`)을 `tar`로 압축하여 업로드하고, 이후 정리합니다. `entrypoint`는 Qiskit Serverless가 실행할 기본 프로그램 실행 파일을 식별합니다. 또한 프로그램에 사용자 정의 `pip` 의존성이 있는 경우 `dependencies` 배열에 추가할 수 있습니다.